In [ ]:
import os
import re
import gc
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path

In [ ]:
def weighted_global_efficiency(G, weight='distance'):
    n = len(G)
    if n < 2:
        return 0.0

    total = 0.0
    for node in G.nodes():
        # Shortest path lengths using the distance attribute
        lengths = nx.single_source_dijkstra_path_length(G, node, weight=weight)
        total += sum(1.0 / d for target, d in lengths.items() if target != node and d > 0)

    return total / (n * (n - 1))

def compute_features(mat, ratio_thres):
    G = nx.from_numpy_array(mat)
    
    # Group 1: features extracted from the original connectivity matrix
    cluster_coeff = nx.clustering(G, weight='weight') 
    strengths = dict(G.degree(weight='weight'))
    
    # Eigenvector centrality
    try:
        eigen_central = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
    except Exception:
        eigen_central = {node: 0.0 for node in G.nodes()}
    D = np.where(mat > 0, 1/mat, 0)
    D = nx.from_numpy_array(D)
    if not nx.is_connected(D):
        path_length = np.nan
    else:
        path_length = nx.average_shortest_path_length(D, weight='weight')
    
    # Group 2: Thresholding matrix
    thres = ratio_thres * mat.max()
    mat_ = np.where(mat > thres, mat, 0)
    G_ = nx.from_numpy_array(mat_)
    between_central = nx.betweenness_centrality(G_, weight='weight', normalized=True)
    dens = nx.density(G_)
    wge = weighted_global_efficiency(G_, weight='distance')

    
    df_features = pd.DataFrame({
        'Node': list(cluster_coeff.keys()),
        'Clustering_Coeff': list(cluster_coeff.values()),
        'Strength': [strengths[node] for node in cluster_coeff.keys()],
        'Eigen_Centrality': [eigen_central[node] for node in cluster_coeff.keys()],
        'Betweenness_Centrality': [between_central[node] for node in cluster_coeff.keys()],
        'Density': dens,
        'Weighted_Global_Efficiency': wge,
        'Characteristic_Path_Length':path_length
    })
    return df_features

# Collect all features and organize them in a dataframe
def compute_all_features(pli_current, ratio_thres):
    
    df_current = compute_features(pli_current, ratio_thres)
    
    df_curr_indexed = df_current.set_index('Node')
    feature_cols = [col for col in df_current.columns if col != 'Node']
    
    df_result = pd.DataFrame(df_curr_indexed)
    
    series_flat = df_result.unstack()
    
    flat_dict = {}
    for (feat_name, node_id), val in series_flat.items():
        flat_dict[f"Node_{node_id}_{feat_name}"] = val
        
    return flat_dict

In [ ]:
def extract_graph_features(folder_path,step=None,ratio_thres=0.7):
    
    folder = Path(folder_path)
    files = sorted([f for f in folder.glob('*.npy')])
    
    if not files:
        return pd.DataFrame()

    rows_list = []
    for file_path in files:
        
        band = file_path.name.split('.')[0]            # 'alpha'
        print('Band: ', band)
        sz = file_path.parent.parent.name
        matrices = np.load(file_path)
                
        for epoch_idx, matrix in enumerate(matrices):
            feat_dict = compute_all_features(matrix, ratio_thres)
            row_data = {
                    'SeizureID': sz,
                    'Band': band,
                    'Epoch': epoch_idx+step
                }
            print('Epoch', epoch_idx+step)
            row_data.update(feat_dict)
            rows_list.append(row_data)

    final_df = pd.DataFrame(rows_list)
    return final_df, matrices.shape[0]

In [ ]:
# ==========================================================================
# THỰC THI CHÍNH (MAIN EXECUTION)
# ==========================================================================
mode = 'PLI' # mode = 'wPLI'
if __name__ == '__main__':
    base_path = Path(f'/kaggle/input/datasets/ngoclannguyen00/chb-mit-connectivity-testing/{mode}_matrix')
    output_dir = Path(f'{mode}_Features_Nodal_Thres_0.5')

    for chb_folder in sorted(base_path.iterdir()):
        if not chb_folder.is_dir():
            continue
            
        num_chb = chb_folder.name  # 'chb01', 'chb02'...
        chb_output = output_dir
        chb_output.mkdir(parents=True, exist_ok=True)
        print(f'Processing {num_chb}...')

        df_list = []
        for sz_path in chb_folder.glob('*'):
            
            sz = sz_path.name
            print('Seizure: ', sz)
            prev_batch_size = 0
            step=0
            for batch_path in sorted(sz_path.glob('*')):
                step += prev_batch_size
                if batch_path.is_dir():
                    features, prev_batch_size = extract_graph_features(batch_path, step=step, ratio_thres=0.5)
                    if not features.empty:
                        df_list.append(features)
                
        if df_list:
            df_result = pd.concat(df_list, ignore_index=True)
            save_path = chb_output / f'{num_chb}.csv'
            df_result.to_csv(save_path, index=False)
            print(f'  [OK] Completed {num_chb} -> Saved to {save_path}')
                
            del df_list, df_result
        gc.collect()